In [7]:
import pymupdf4llm

extract_data=pymupdf4llm.to_markdown("Botany.pdf",
                                     write_images=True,
                                     image_path="botany_images/",
                                     image_format="png",
                                     dpi=100
                                    )

In [18]:
import os
import json
import time
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
api_key=os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("Gemini key not found")

client=genai.Client(api_key=api_key)

images="botany_images"
output_json="botany_captions.json"
output_md="botany_captions.md"

prompt=  """Look at this image extracted from a Botany textbook.

If it is a real diagram or illustration, write a concise 1-2 sentence caption that acts as a quick reference. 

Your caption MUST include:
1. The exact botanical subject (e.g., "Plant cell structure", "Leaf anatomy", "Root system", "Photosynthesis cycle").
2. The specific parts or labels shown (e.g., "showing the xylem, phloem, and stomata").

Do NOT describe the colors, shapes, or background. Do not summarize the surrounding text. Keep it factual and brief.
If it is a fragment (a single word, a tiny line, or meaningless shapes), reply exactly with: FRAGMENT    """

print("Processing botany images")
files=sorted([f for f in os.listdir(images) if f.lower().endswith(".png")])
dataset=[]

for fname in files:
    img_path=os.path.join(images, fname)
   

    with open(img_path, "rb") as f:
        img_bytes=f.read()

    ext=fname.lower().rsplit(".",1)[-1]
    mime="image/png" if ext=="png" else "image/jpeg"

    try:
        response=client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=[prompt, types.Part.from_bytes(data=img_bytes, mime_type=mime)]
        )

        caption=response.text.strip()

        if "Fragment" in caption:
            print(f"Fragmented {fname}.",end=" ")
        else:
 
            dataset.append({"image_file":fname, 
                           "caption":caption
                           })
    except Exception as e:
        print(f"Error: {e}")

    time.sleep(4)

with open(output_json, "w", encoding="UTF-8") as f:
    json.dump(dataset, f, indent=2)
print("Created json file")

with open(output_md, "w", encoding="UTF-8") as f:
    f.write("Image Captioning")
    f.write(f"Total images, {len(dataset)}\n")
    f.write("\n\n\n")

    for items in dataset:
        f.write(f"![{items['image_file']}]({images}/{items['image_file']})\n\n")
        f.write(f"Captions: {items['caption']}\n\n")
        f.write("\n")
print("Created md file")
                                                                                                                                                                                                

Processing botany images
Created json file
Created md file
